# Parte 1
### Carga datos previamente limpiados y separar las características (X) de la variable objetivo (Y)

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# definir ruta del dataset limpio
ruta_datos = os.path.join("..", "data", "procesado", "datos_estudiantes_limpios.csv")

# cargar datos procesados
df = pd.read_csv(ruta_datos, sep=";", decimal=",")

# codificar variables categoricas a tipo category para lightgbm
df["dependencia"] = df["dependencia"].astype("category")
df["estrato_analitico"] = df["estrato_analitico"].astype("category")

# separar caracteristicas y variable objetivo
X = df.drop(columns=["aprendizaje_con_tecnologia"])
y = df["aprendizaje_con_tecnologia"]

# dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"dimensiones de entrenamiento: {X_train.shape}")
print(f"dimensiones de prueba: {X_test.shape}")

dimensiones de entrenamiento: (8260, 8)
dimensiones de prueba: (2066, 8)


# Parte 2

In [ ]:
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error 

# transformar variables categoricas a codigos numericos
for col in ["dependencia", "estrato_analitico"]:
    X_train[col] = X_train[col].cat.codes
    X_test[col] = X_test[col].cat.codes

# configurar hiperparametros basicos del modelo
params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "random_state": 42,
    "verbose": -1
}

# entrenar el modelo de caja negra
modelo = lgb.LGBMRegressor(**params)
modelo.fit(X_train, y_train)

# realizar predicciones en el conjunto de prueba
y_pred = modelo.predict(X_test)

# calcular metricas usando root mean squared error
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred) 

print(f"error cuadratico medio (rmse): {rmse:.4f}")
print(f"coeficiente de determinacion (r2): {r2:.4f}")
print(f"error absoluto medio (mae): {mae:.4f}") 

error cuadratico medio (rmse): 0.4726
coeficiente de determinacion (r2): 0.4269


# Parte 3
### Guardar el modelo

In [ ]:
import joblib

# asegurar existencia de la carpeta de modelos
ruta_modelos = os.path.join("..", "resultados", "modelos")
os.makedirs(ruta_modelos, exist_ok=True)

# guardar el modelo entrenado en formato pkl
joblib.dump(modelo, os.path.join(ruta_modelos, "modelo_lightgbm.pkl"))
print("modelo de caja negra guardado exitosamente")

modelo de caja negra guardado exitosamente
